In [1]:
import joblib

In [7]:
from tqdm import tqdm

In [2]:
joblib.load("models/xgb-kfold-binary-13/model.pth")

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.001, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=15, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
import pandas as pd
import numpy as np
import joblib

# Load model and artifacts
model = joblib.load("models/xgb-kfold-binary-13/model.pth")
artifacts = joblib.load("/mnt/object/train/transform_artifacts.pkl")

KEEP_COLS = artifacts["keep_cols"]

CATEGORICAL_ONEHOT = artifacts["categorical_onehot"]
CATEGORICAL_LABEL = artifacts["categorical_label"]
median_values = artifacts["median_values"]
mode_values = artifacts["mode_values"]
label_encoders = artifacts["label_encoders"]
onehot_columns_train = artifacts["onehot_columns"]
scaler = artifacts["scaler"]
numeric_cols = artifacts["numeric_cols"]
LABEL_COL = "risk_level"

# --- Helper functions ---

def parse_txt_to_df(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    data = {}
    for line in lines:
        if ":" in line:
            k, v = line.strip().split(":", 1)
            try:
                data[k.strip()] = float(v.strip())
            except ValueError:
                data[k.strip()] = v.strip()
    return pd.DataFrame([data])

def transform_input_df(df):
    # Load transform artifacts
    # artifacts = joblib.load("/mnt/data/LoanData/train/transform_artifacts.pkl")
    # KEEP_COLS = artifacts["keep_cols"]
    # median_values = artifacts["median_values"]
    # mode_values = artifacts["mode_values"]
    # label_encoders = artifacts["label_encoders"]
    # onehot_columns_train = artifacts["onehot_columns"]
    # scaler = artifacts["scaler"]
    # numeric_cols = artifacts["numeric_cols"]
    # CATEGORICAL_ONEHOT = artifacts["categorical_onehot"]
    # CATEGORICAL_LABEL = artifacts["categorical_label"]

    log_msgs = []

    df = df[KEEP_COLS]
    label_series = df.pop(LABEL_COL).map(lambda x: 0 if x == "Low" else 1)

    # Fill NaNs using training stats
    for col in tqdm(df.columns, desc="Handling NaNs with training stats"):
        if df[col].isnull().sum() > 0:
            if col in mode_values:
                df[col] = df[col].fillna(mode_values[col])
            elif col in median_values:
                df[col] = df[col].fillna(median_values[col])
            else:
                df[col] = df[col].fillna(0)

    # One-hot encoding
    for col in CATEGORICAL_ONEHOT:
        dummies = pd.get_dummies(df[col], prefix=col)
    
        # Filter relevant onehot columns for current variable
        relevant_cols = [c for c in onehot_columns_train if c.startswith(f"{col}_")]
    
            # Add missing dummy columns
        for dummy_col in relevant_cols:
            if dummy_col not in dummies.columns:
                dummies[dummy_col] = 0

        dummies = dummies[relevant_cols]
        df = pd.concat([df.drop(columns=[col]), dummies], axis=1)


    # Label encoding
    for col in CATEGORICAL_LABEL:
        le = label_encoders[col]
        df[col] = le.transform(df[col].astype(str))

    # Standard scaling
    df[numeric_cols] = scaler.transform(df[numeric_cols])
    df[LABEL_COL] = label_series.reset_index(drop=True)

    return df
    

# --- Main execution ---

# Replace with your actual .txt file path
txt_file_path = r"sample_5.txt"

raw_df = parse_txt_to_df(txt_file_path)
print("📦 Raw extracted features:")
display(raw_df)

true_label = transformed_df[LABEL_COL].iloc[0] if LABEL_COL in raw_df.columns else None
# if LABEL_COL in raw_df.columns:
#     raw_df = raw_df.drop(columns=[LABEL_COL])

transformed_df = transform_input_df(raw_df)
if LABEL_COL in transformed_df.columns:
    transformed_df = transformed_df.drop(columns=[LABEL_COL])
print(f"\n🧪 Transformed shape: {transformed_df.shape}")
display(transformed_df)

# Predict
prediction = model.predict(transformed_df.values)[0]
print(f"\n🔮 Predicted risk class: {prediction}")
print(f"✅ True label (if present): {true_label}")


📦 Raw extracted features:


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,sub_grade,home_ownership,annual_inc,verification_status,dti,...,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,risk_level
0,17000.0,17000.0,17000.0,17.47,426.81,D1,MORTGAGE,80000.0,Source Verified,6.29,...,2.0,83.3,0.0,0.0,0.0,30376.0,10078.0,10300.0,10276.0,Low


Handling NaNs with training stats: 100%|██████████| 61/61 [00:00<00:00, 7434.55it/s]


🧪 Transformed shape: (1, 68)


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,sub_grade,annual_inc,dti,delinq_2yrs,fico_range_low,...,total_il_high_credit_limit,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified
0,0.212017,0.21264,0.214529,0.906407,-0.071484,0.749656,0.016062,-0.891612,-0.354305,-1.017377,...,-0.746058,0,True,0,0,0,0,0,True,0



🔮 Predicted risk class: 0
✅ True label (if present): 0


In [9]:
print(transformed_df.columns)

Index(['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate',
       'installment', 'sub_grade', 'annual_inc', 'dti', 'delinq_2yrs',
       'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt',
       'tot_cur_bal', 'open_act_il', 'open_il_12m', 'total_bal_il',
       'open_rv_12m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi',
       'total_cu_tl', 'inq_last_12m', 'avg_cur_bal', 'bc_open_to_buy',
       'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mort_acc',
       'mths_since_recent_inq', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', '

In [10]:
print(raw_df.columns)

Index(['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate',
       'installment', 'sub_grade', 'home_ownership', 'annual_inc',
       'verification_status', 'dti', 'delinq_2yrs', 'fico_range_low',
       'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'collections_12_mths_ex_med',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_act_il',
       'open_il_12m', 'total_bal_il', 'open_rv_12m', 'max_bal_bc', 'all_util',
       'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths',
       'delinq_amnt', 'mort_acc', 'mths_since_recent_inq', 'num_actv_bc_tl',
       'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl',
       'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats',
       'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
       'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75',
    